[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/yildiztu/Code-101-Problems-Logistics-SCM/blob/main/Part%20I%20-%20The%20Port%20%26%20Container%20Terminal%20%28Problems%201-46%29/B.%20Core%20Quay-Side%20Operations%20%28The%20Ship-to-Shore%20Interface%29/08.%20The%20Quay%20Crane%20Assignment%20Problem/P8-Tier-1_executed.ipynb)

---


# 8. The Quay Crane Assignment Problem
## Tier 1 — Scenario-Based Stochastic Programming Formulation

### Goal
This notebook uses exhaustive enumeration to solve a small, scenario-based Quay Crane Assignment Problem (QCAP). A fixed assignment of cranes to vessels is evaluated under uncertain productivity conditions, and the assignment with the smallest probability-weighted total vessel completion time is selected.

### Inputs and outputs
The inputs are vessel workloads, minimum and maximum crane requirements, crane productivity rates for each scenario, scenario probabilities, and a productivity interference factor. The outputs are the feasible crane assignments, their scenario-specific completion times, the assignment with minimum expected total completion time, and a sensitivity analysis of the objective as the delay probability changes.

### Modeling assumptions and scope
- Vessel workloads and crane-count requirements are deterministic.
- Scenario productivity rates are known, and scenario probabilities sum to one.
- The crane assignment is selected before the productivity scenario is known; the same assignment is used in every scenario. There are no scenario-dependent reassignment decisions.
- Every available crane is assigned to exactly one vessel. Each vessel must receive at least its specified minimum and at most its specified maximum number of cranes.
- When a vessel has \(k\) assigned cranes, each assigned crane's productivity is multiplied by \((1-\alpha)^{k-1}\). The adjusted rates are added to obtain that vessel's effective productivity.
- Vessel completion time is workload divided by effective productivity. The objective sums vessel completion times; it is not the time at which the entire quay operation finishes (the maximum vessel completion time).
- The model does not represent vessel-specific crane positions, travel or setup times, precedence constraints, time-varying workloads, or interactions between vessels.

### Solution sequence
1. Define the instance data and scenario probabilities.
2. Formulate scenario-specific completion times and the expected-total-time objective.
3. Enumerate all assignments and retain those that satisfy the crane-count requirements.
4. Evaluate every feasible assignment under every scenario and select a minimum-objective assignment.
5. Inspect the results and examine how the optimal objective responds to changes in the delay probability.

The example contains three vessels, four cranes, and two productivity scenarios: normal operations and equipment delays.

## Step 1: Problem Setup and Parameters

The instance data define the workload to be served, the permitted crane-count range for each vessel, and the productivity of every vessel–crane pair in each scenario. Productivity is measured in TEUs per hour, so workload divided by effective productivity gives completion time in hours.

In [ ]:
import numpy as np
import pandas as pd
from typing import Dict, List, Tuple

# These dimensions make the scenario data and assignment representation explicit.
NUM_VESSELS = 3
NUM_CRANES = 4
NUM_SCENARIOS = 2

# Vessel data: workload in TEUs, permitted crane counts, and display name.
vessels = {
    0: {'workload': 800, 'min_cranes': 1, 'max_cranes': 2, 'name': 'V1'},
    1: {'workload': 1200, 'min_cranes': 1, 'max_cranes': 2, 'name': 'V2'},
    2: {'workload': 600, 'min_cranes': 1, 'max_cranes': 2, 'name': 'V3'}
}

# Scenario probabilities: normal operations (70%) and equipment delays (30%).
scenario_probs = [0.7, 0.3]

# Productivity rates are in TEUs per hour; rows are vessels and columns are cranes.
productivity_normal = np.array([
    [25, 27, 24, 26],
    [23, 25, 26, 24],
    [26, 24, 25, 27]
])

productivity_delayed = np.array([
    [20, 22, 19, 21],
    [18, 20, 21, 19],
    [21, 19, 20, 22]
])

productivity_by_scenario = [productivity_normal, productivity_delayed]

# Each additional crane on the same vessel applies another interference factor.
interference_factor = 0.08

assert len(vessels) == NUM_VESSELS
assert len(scenario_probs) == NUM_SCENARIOS
assert np.isclose(sum(scenario_probs), 1.0)
assert all(matrix.shape == (NUM_VESSELS, NUM_CRANES)
           for matrix in productivity_by_scenario)

print("Problem setup complete")
print(f"Vessels: {NUM_VESSELS}, Cranes: {NUM_CRANES}, Scenarios: {NUM_SCENARIOS}")
print(f"Scenario probabilities: {scenario_probs}")

## Step 2: Mathematical Model Formulation

### Intuition and decisions
A crane assignment determines which productivity rates contribute to each vessel's service rate. Because the assignment is made before the uncertain scenario is known, it must be the same in every scenario. Completion times can differ by scenario because the productivity rates differ.

Let \(V=\{0,\ldots,\texttt{NUM\_VESSELS}-1\}\) be the vessel set, \(C=\{0,\ldots,\texttt{NUM\_CRANES}-1\}\) the crane set, and \(S=\{0,\ldots,\texttt{NUM\_SCENARIOS}-1\}\) the scenario set. The binary decision variable \(x_{vc}\) is 1 if crane \(c\) is assigned to vessel \(v\), and 0 otherwise. The scenario-dependent quantity \(T_{vs}\) is the completion time of vessel \(v\) in scenario \(s\). The assignment is a here-and-now decision; \(T_{vs}\) is calculated for each scenario and does not represent a separate assignment decision.

### Assignment constraints
Every crane is assigned to exactly one vessel, and each vessel receives a number of cranes within its specified range:

$$
\sum_{v\in V} x_{vc}=1 \qquad \forall c\in C,
$$

$$
L_v \leq \sum_{c\in C}x_{vc} \leq U_v \qquad \forall v\in V,
$$

$$
x_{vc}\in\{0,1\} \qquad \forall v\in V,\;c\in C.
$$

Here, \(L_v\) and \(U_v\) are the minimum and maximum crane counts for vessel \(v\), stored as `min_cranes` and `max_cranes` in `vessels`. The first constraint makes assignments mutually exclusive across vessels and ensures that no crane is left unused.

### Productivity, interference, and completion time
Let \(r_{vcs}\) be the productivity of crane \(c\) on vessel \(v\) in scenario \(s\), and let \(\alpha\) be the interference factor. If \(k_v=\sum_{c\in C}x_{vc}\) cranes serve vessel \(v\), the effective productivity and completion time are

$$
P_{vs}(x)=\sum_{c\in C} r_{vcs}x_{vc}(1-\alpha)^{k_v-1},
\qquad
T_{vs}(x)=\frac{w_v}{P_{vs}(x)}.
$$

The workload \(w_v\) is stored as `workload`; \(r_{vcs}\) comes from `productivity_by_scenario`; and \(\alpha\) is `interference_factor`. Feasible assignments give every vessel at least one crane, so the denominator is positive. In `calculate_completion_time`, the assigned rates are multiplied by the common factor \((1-\alpha)^{k_v-1}\), summed, and divided into the vessel workload.

### Expected-total-time objective
If \(p_s\) is the probability of scenario \(s\), the objective is

$$
\min_x \sum_{s\in S}p_s\sum_{v\in V}T_{vs}(x).
$$

The probabilities are `scenario_probs`. This objective is the expected value of the sum of vessel completion times: first sum the vessel times within each scenario, then weight each scenario total by its probability. This finite-scenario formulation is evaluated by exhaustive enumeration in this notebook; no mathematical-programming solver is used.

In [ ]:
def calculate_completion_time(
    vessel_idx: int,
    assigned_cranes: List[int],
    scenario_idx: int,
    vessels: Dict[int, dict],
    productivity_by_scenario: List[np.ndarray],
    interference_factor: float
) -> float:
    """Calculate a vessel's completion time for an assignment and scenario.

    Args:
        vessel_idx: Index of the vessel being evaluated.
        assigned_cranes: Crane indices assigned to that vessel.
        scenario_idx: Index of the productivity scenario.
        vessels: Vessel workloads and crane-count requirements.
        productivity_by_scenario: Productivity matrices indexed by scenario.
        interference_factor: Per-additional-crane productivity reduction.

    Returns:
        Completion time in hours.
    """
    if not assigned_cranes:
        return float('inf')

    workload = vessels[vessel_idx]['workload']
    rates = productivity_by_scenario[scenario_idx][vessel_idx, assigned_cranes]

    # Each crane experiences interference from the other cranes on this vessel.
    num_cranes = len(assigned_cranes)
    interference_penalty = (1 - interference_factor) ** (num_cranes - 1)
    total_productivity = sum(rates * interference_penalty)

    return workload / total_productivity


# This check makes the units and interference calculation tangible for one assignment.
test_time = calculate_completion_time(
    0, [0, 1], 0, vessels, productivity_by_scenario, interference_factor
)
print(f"V1 completion time with cranes C1 and C2 (normal): {test_time:.2f} hours")

test_time_delayed = calculate_completion_time(
    0, [0, 1], 1, vessels, productivity_by_scenario, interference_factor
)
print(f"V1 completion time with cranes C1 and C2 (delayed): {test_time_delayed:.2f} hours")

## Step 3: Brute-Force Solution Approach

Exhaustive enumeration checks each possible assignment of the available cranes and keeps only those that satisfy every vessel's minimum and maximum crane requirements. With \(|V|\) vessels and \(|C|\) cranes, there are \(|V|^{|C|}\) raw mappings from cranes to vessels. Each mapping can be checked by counting the cranes assigned to each vessel. This direct search is transparent and practical for this small instance, but the number of mappings grows rapidly as the instance grows.

The function below returns assignments as dictionaries from vessel indices to lists of crane indices. Because each crane is assigned one vessel during enumeration, no crane can appear in two lists and all cranes are assigned.

In [ ]:
def generate_feasible_assignments(
    vessels: Dict[int, dict],
    num_cranes: int
) -> List[Dict[int, List[int]]]:
    """Enumerate all assignments satisfying vessel crane-count bounds.

    Each raw mapping assigns every crane to exactly one vessel. The count
    bounds then determine whether that mapping is feasible.
    """
    from itertools import product

    vessel_indices = list(vessels)
    feasible_assignments = []

    # A tuple records the vessel receiving each crane; exhaustive enumeration
    # keeps the search logic visible for this small teaching instance.
    for crane_destinations in product(vessel_indices, repeat=num_cranes):
        assignment = {vessel_idx: [] for vessel_idx in vessel_indices}
        for crane_idx, vessel_idx in enumerate(crane_destinations):
            assignment[vessel_idx].append(crane_idx)

        satisfies_crane_bounds = all(
            vessels[vessel_idx]['min_cranes'] <= len(cranes)
            <= vessels[vessel_idx]['max_cranes']
            for vessel_idx, cranes in assignment.items()
        )
        if satisfies_crane_bounds:
            feasible_assignments.append(assignment)

    return feasible_assignments


feasible_assignments = generate_feasible_assignments(vessels, NUM_CRANES)
assert feasible_assignments

for assignment in feasible_assignments:
    assigned_cranes = [
        crane_idx
        for crane_list in assignment.values()
        for crane_idx in crane_list
    ]
    assert sorted(assigned_cranes) == list(range(NUM_CRANES))
    assert all(
        vessels[vessel_idx]['min_cranes'] <= len(cranes)
        <= vessels[vessel_idx]['max_cranes']
        for vessel_idx, cranes in assignment.items()
    )

print(f"Generated {len(feasible_assignments)} feasible crane assignments")
for assignment_idx, assignment in enumerate(feasible_assignments[:3], start=1):
    print(f"Assignment {assignment_idx}: {assignment}")

## Step 4: Evaluate All Assignments and Find an Optimal Solution

For each feasible assignment, calculate every vessel's completion time in every scenario. The expected total is the probability-weighted sum of the scenario totals, matching the objective in Step 2. Since the assignment set is finite, choosing the smallest evaluated objective gives a global optimum for this enumerated model. If multiple assignments have numerically equal objective values, the code identifies all of them using a floating-point tolerance.

In [ ]:
def evaluate_assignment(
    assignment: Dict[int, List[int]],
    vessels: Dict[int, dict],
    productivity_by_scenario: List[np.ndarray],
    interference_factor: float,
    scenario_probs: List[float]
) -> Tuple[float, List[float], List[float]]:
    """Return expected total time and vessel times for both scenarios.

    The assignment is fixed across scenarios, while each scenario uses its
    own productivity matrix and probability.
    """
    if len(productivity_by_scenario) != len(scenario_probs):
        raise ValueError("Each scenario must have one probability.")
    if not np.isclose(sum(scenario_probs), 1.0):
        raise ValueError("Scenario probabilities must sum to one.")

    scenario_times = []
    for scenario_idx in range(len(productivity_by_scenario)):
        times_for_scenario = []
        for vessel_idx, assigned_cranes in assignment.items():
            completion_time = calculate_completion_time(
                vessel_idx,
                assigned_cranes,
                scenario_idx,
                vessels,
                productivity_by_scenario,
                interference_factor
            )
            times_for_scenario.append(completion_time)
        scenario_times.append(times_for_scenario)

    expected_total = sum(
        probability * sum(times_for_scenario)
        for probability, times_for_scenario in zip(scenario_probs, scenario_times)
    )

    return expected_total, scenario_times[0], scenario_times[1]


assignment_results = []
for assignment_idx, assignment in enumerate(feasible_assignments):
    expected_time, times_normal, times_delayed = evaluate_assignment(
        assignment,
        vessels,
        productivity_by_scenario,
        interference_factor,
        scenario_probs
    )
    assignment_results.append({
        'assignment_id': assignment_idx,
        'assignment': assignment,
        'expected_total_time': expected_time,
        'normal_times': times_normal,
        'delayed_times': times_delayed
    })

best_expected_time = min(
    result['expected_total_time'] for result in assignment_results
)
optimal_results = [
    result for result in assignment_results
    if np.isclose(
        result['expected_total_time'],
        best_expected_time,
        rtol=1e-9,
        atol=1e-9
    )
]
optimal_result = optimal_results[0]

print(f"Evaluated {len(assignment_results)} feasible assignments")
print(
    f"\nMinimum expected total completion time: "
    f"{best_expected_time:.2f} hours"
)
print(f"Number of assignments tied within tolerance: {len(optimal_results)}")
for result_idx, result in enumerate(optimal_results, start=1):
    print(f"\nOptimal assignment {result_idx}:")
    for vessel_idx, cranes in result['assignment'].items():
        vessel_name = vessels[vessel_idx]['name']
        crane_names = [f'C{crane_idx + 1}' for crane_idx in cranes]
        print(f"  {vessel_name}: {crane_names}")

print("\nVessel completion times for the first optimal assignment:")
print(f"  Normal scenario ({scenario_probs[0]:.0%}): "
      f"{[f'{time:.2f} h' for time in optimal_result['normal_times']]}")
print(f"  Delayed scenario ({scenario_probs[1]:.0%}): "
      f"{[f'{time:.2f} h' for time in optimal_result['delayed_times']]}")

## Step 5: Detailed Results Analysis

The table ranks assignments by expected total completion time and displays each assignment's total completion time in the normal and delayed scenarios. The histogram summarizes the distribution of objective values across feasible assignments; its vertical reference line marks the minimum. Vessel completion times within a scenario are summed for this comparison, consistent with the stated objective.

In [ ]:
import matplotlib.pyplot as plt

# Sort by the objective so that the displayed rows are the best-ranked assignments.
sorted_results = sorted(
    assignment_results,
    key=lambda result: result['expected_total_time']
)
results_table = []
for result in sorted_results[:10]:
    results_table.append({
        'Assignment': str(result['assignment']),
        'Expected total (hours)': f"{result['expected_total_time']:.2f}",
        'Normal total (hours)': f"{sum(result['normal_times']):.2f}",
        'Delayed total (hours)': f"{sum(result['delayed_times']):.2f}"
    })

df_results = pd.DataFrame(results_table)
print("Best-ranked assignment results:")
print(df_results.to_string(index=False))

finite_expected_times = [
    result['expected_total_time']
    for result in assignment_results
    if np.isfinite(result['expected_total_time'])
]

if finite_expected_times:
    plt.figure(figsize=(10, 6))
    plt.hist(finite_expected_times, bins=20, alpha=0.7, edgecolor='black')
    plt.axvline(
        best_expected_time,
        color='red',
        linestyle='--',
        linewidth=2,
        label=f'Minimum: {best_expected_time:.2f} h'
    )
    plt.xlabel('Expected total vessel completion time (hours)')
    plt.ylabel('Number of feasible assignments')
    plt.title('Expected Total Completion Times Across Feasible Assignments')
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.show()

    worst_expected_time = max(finite_expected_times)
    print(
        f"\nExpected-time range: {min(finite_expected_times):.2f} to "
        f"{worst_expected_time:.2f} hours"
    )
    if worst_expected_time > 0:
        improvement = (
            (worst_expected_time - best_expected_time)
            / worst_expected_time
            * 100
        )
        print(f"Minimum relative to the worst assignment: {improvement:.1f}% lower")
else:
    print("No finite expected completion times are available to plot.")

## Step 6: What-If Analysis — Sensitivity to Scenario Probabilities

The scenario productivity matrices remain fixed in this experiment; only the probabilities change. For each tested delay probability \(q\), the normal-scenario probability is \(1-q\), and each feasible assignment is evaluated using

$$
(1-q)\sum_{v\in V}T_{v,0}(x)+q\sum_{v\in V}T_{v,1}(x).
$$

The minimum over feasible assignments is then recorded. This analysis shows how the optimal objective responds to different beliefs about delay likelihood. It does not change the input data or the assignment constraints.

In [ ]:
delay_probs = [0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9]
optimal_times_by_prob = []
optimal_assignments_by_prob = []

for delay_prob in delay_probs:
    probabilities = [1 - delay_prob, delay_prob]
    results_for_probability = []

    for assignment in feasible_assignments:
        expected_time, _, _ = evaluate_assignment(
            assignment,
            vessels,
            productivity_by_scenario,
            interference_factor,
            probabilities
        )
        results_for_probability.append((expected_time, assignment))

    best_time = min(time for time, _ in results_for_probability)
    best_assignments = [
        assignment
        for time, assignment in results_for_probability
        if np.isclose(time, best_time, rtol=1e-9, atol=1e-9)
    ]
    optimal_times_by_prob.append(best_time)
    optimal_assignments_by_prob.append(best_assignments)

plt.figure(figsize=(10, 6))
plt.plot(delay_probs, optimal_times_by_prob, 'bo-', linewidth=2, markersize=8)
plt.xlabel('Probability of equipment delays')
plt.ylabel('Minimum expected total completion time (hours)')
plt.title('Sensitivity of the Minimum Expected Time to Delay Probability')
plt.grid(True, alpha=0.3)
plt.axvline(
    scenario_probs[1],
    color='red',
    linestyle='--',
    alpha=0.7,
    label=f"Base delay probability ({scenario_probs[1]:.0%})"
)
plt.legend()
plt.show()

print("\nSensitivity analysis results:")
for probability, time, assignments in zip(
    delay_probs,
    optimal_times_by_prob,
    optimal_assignments_by_prob
):
    print(
        f"Delay probability {probability:.1f}: minimum expected time = "
        f"{time:.2f} hours; tied optimal assignments = {len(assignments)}"
    )

print(
    f"\nMinimum expected-time range: {min(optimal_times_by_prob):.2f} to "
    f"{max(optimal_times_by_prob):.2f} hours"
)

## Results Interpretation and Key Insights

### Interpreting the assignment results
The selected assignment minimizes the probability-weighted sum of vessel completion times among the feasible assignments. Its normal- and delayed-scenario vessel times show how the same crane allocation performs under the two productivity conditions. The objective is an expectation over scenarios, not a guarantee that the realized total will equal that expectation.

If multiple assignments have objective values equal within the stated numerical tolerance, each is optimal for the model. The enumeration order determines which tied assignment is displayed first; it does not make that assignment uniquely optimal.

### Interpreting the sensitivity analysis
Each point on the sensitivity plot is the smallest expected total completion time at the specified delay probability. Scenario completion times are fixed for each assignment, so changing the probability changes the weight placed on each scenario. The reported optimal objective does not, by itself, indicate whether the crane assignment changes; the number of tied optimal assignments is reported for each tested probability.

### Model checks and limitations
The enumeration checks the crane-count bounds and assigns every crane exactly once. Completion-time calculations use the same workload, productivity, and interference definitions as the mathematical model. The objective sums vessel completion times and therefore should not be interpreted as the quay's makespan. The exhaustive search is appropriate for this small teaching instance; its raw search space grows as \(|V|^{|C|}\), so larger instances require methods beyond the scope of this notebook.

This scenario-based formulation illustrates how uncertain operating conditions can be incorporated into an assignment decision while preserving a clear, auditable solution process.